In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl

from memory_profiler import profile

pd.set_option('display.max_columns', None)

In [ ]:
# @profile
def process_pandas(path: str):
    """
    Reads a path for a CSV file with pandas and return a DataFrame object after executing some processing.
    """
    df = pd.read_csv(path)
    grouped_df = (
        df.query("passenger_count > 1")
        .groupby(by="PULocationID")
        .agg({"fare_amount": "mean"})
        .sort_values(by="fare_amount", ascending=False)
        .reset_index()
    )

    return grouped_df

In [ ]:
# @profile
def process_polars(path: str):
    """
    Reads a path for a CSV file with polars and return a DataFrame object after executing some processing.
    """

    df = pl.scan_csv(path)
    df = df.filter(pl.col("passenger_count") > 1)
    grouped_df = (
        df.group_by("PULocationID")
        .agg(pl.col("fare_amount").mean())
        .sort(by="fare_amount", descending=True)
        .collect()
    )

    return grouped_df

In [ ]:
path = 'data/test_data.csv'

In [ ]:
%%time
df = process_pandas(path=path)
display(df)

In [ ]:
%%time
df = process_polars(path=path)
display(df)

In [ ]:
import matplotlib.pyplot as plt

# =====================================================================
# 1. INSIRA AQUI OS SEUS RESULTADOS REAIS MEDIDOS NO BENCHMARK
# =====================================================================
pandas_time = 16.5   # Tempo do Pandas em segundos
polars_time = 2.77    # Tempo do Polars em segundos

pandas_memory = 3200  # Pico de memória do Pandas em MB
polars_memory = 450   # Pico de memória do Polars em MB
# =====================================================================

# Configuração dos dados para o plot
labels = ['Pandas (Eager)', 'Polars (Lazy)']
times = [pandas_time, polars_time]
memories = [pandas_memory, polars_memory]

# Criando a figura com 2 gráficos lado a lado (1 linha, 2 colunas)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Cor padrão: Azul para Pandas, Laranja para Polars (cores oficiais das libs)
cores = ['#150458', '#FF7A00'] # Azul escuro (Pandas/Python) e Laranja (Polars/Rust)

# --- Gráfico 1: Tempo de Execução ---
bars1 = ax1.bar(labels, times, color=cores)
ax1.set_title('Tempo de Execução (Menor é Melhor)', fontsize=12)
ax1.set_ylabel('Segundos')
ax1.set_ylim(0, max(times) * 1.15) # Dá um espaço extra no topo para o texto

# Adicionando os valores exatos no topo das barras
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + (max(times)*0.02),
             f'{yval:.1f}s', ha='center', va='bottom', fontweight='bold')

# --- Gráfico 2: Pico de Memória RAM ---
bars2 = ax2.bar(labels, memories, color=cores)
ax2.set_title('Pico de Memória RAM (Menor é Melhor)', fontsize=12)
ax2.set_ylabel('Megabytes (MB)')
ax2.set_ylim(0, max(memories) * 1.15)

# Adicionando os valores exatos no topo das barras
for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + (max(memories)*0.02),
             f'{yval} MB', ha='center', va='bottom', fontweight='bold')

# Título geral da figura
plt.suptitle('Benchmark: Pandas vs Polars (Processamento de >1GB)', fontsize=14, fontweight='bold')

# Ajusta o layout para não cortar textos
plt.tight_layout()

# Salva o gráfico na pasta local (Excelente para a entrega do desafio!)
plt.savefig('benchmark_resultado.png', dpi=300)

print("Gráfico 'benchmark_resultado.png' gerado com sucesso na pasta atual!")

"Ao reescrevermos nossa ingestão de dados, saímos de um modelo que carregava todo o histórico na memória (Pandas) para um modelo inteligente que só extrai o necessário (Polars). Isso reduz a necessidade de máquinas caras na nuvem (redução de custo de infraestrutura) e diminui o tempo de processamento de horas para minutos, permitindo que a área de negócios tenha análises atualizadas muito mais rápido."